### 1. Analyze Customer Behavior Distribution

Analyze the distribution of key customer behavioral metrics to understand relative customer activity and purchasing behavior.

These distributions provide context for later growth opportunity analysis without relying on arbitrary thresholds.

In [0]:
SELECT
    COUNT(*) AS customers,

    ROUND(AVG(total_spend), 2) AS avg_spend,
    ROUND(PERCENTILE_APPROX(total_spend, 0.25), 2) AS spend_p25,
    ROUND(PERCENTILE_APPROX(total_spend, 0.50), 2) AS spend_p50,
    ROUND(PERCENTILE_APPROX(total_spend, 0.75), 2) AS spend_p75,

    ROUND(AVG(total_baskets), 2) AS avg_baskets,
    PERCENTILE_APPROX(total_baskets, 0.25) AS baskets_p25,
    PERCENTILE_APPROX(total_baskets, 0.50) AS baskets_p50,
    PERCENTILE_APPROX(total_baskets, 0.75) AS baskets_p75,

    ROUND(AVG(avg_basket_value), 2) AS avg_basket_value,
    ROUND(PERCENTILE_APPROX(avg_basket_value, 0.25), 2) AS basket_value_p25,
    ROUND(PERCENTILE_APPROX(avg_basket_value, 0.50), 2) AS basket_value_p50,
    ROUND(PERCENTILE_APPROX(avg_basket_value, 0.75), 2) AS basket_value_p75,

    ROUND(AVG(active_days), 2) AS avg_active_days,
    PERCENTILE_APPROX(active_days, 0.25) AS active_days_p25,
    PERCENTILE_APPROX(active_days, 0.50) AS active_days_p50,
    PERCENTILE_APPROX(active_days, 0.75) AS active_days_p75

FROM workspace.consumer_analytics_gold.customer_360;

customers,avg_spend,spend_p25,spend_p50,spend_p75,avg_baskets,baskets_p25,baskets_p50,baskets_p75,avg_basket_value,basket_value_p25,basket_value_p50,basket_value_p75,avg_active_days,active_days_p25,active_days_p50,active_days_p75
2500,3222.99,970.11,2156.65,4410.1,110.59,39,79,142,31.62,18.32,27.41,40.54,90.21,34,70,124


### 2. Compare Pre-Campaign and Campaign-Period Behavior

Compare purchasing behavior of campaign-assigned households during each campaign with an equally sized period immediately before the campaign.

Using equal observation windows makes spend and shopping frequency more comparable between the two periods.

**BEFORE:** Period immediately preceding the campaign, with the same duration as the campaign.  
**DURING:** Campaign active period.

In [0]:
SELECT
    ch.CAMPAIGN,
    ch.household_key,
    d.START_DAY,
    d.END_DAY,
    d.END_DAY - d.START_DAY + 1 AS campaign_duration_days,

    t.DAY,
    t.BASKET_ID,
    t.SALES_VALUE,

    CASE
        WHEN t.DAY BETWEEN
            d.START_DAY - (d.END_DAY - d.START_DAY + 1)
            AND d.START_DAY - 1
        THEN 'BEFORE'

        WHEN t.DAY BETWEEN d.START_DAY AND d.END_DAY
        THEN 'DURING'

    END AS campaign_period

FROM workspace.consumer_analytics_silver.campaign_households ch

INNER JOIN workspace.consumer_analytics_silver.campaign_desc d
    ON ch.CAMPAIGN = d.CAMPAIGN

INNER JOIN workspace.consumer_analytics_silver.transactions t
    ON ch.household_key = t.household_key

WHERE t.DAY BETWEEN
    d.START_DAY - (d.END_DAY - d.START_DAY + 1)
    AND d.END_DAY

ORDER BY
    ch.CAMPAIGN,
    ch.household_key,
    t.DAY;

CAMPAIGN,household_key,START_DAY,END_DAY,campaign_duration_days,DAY,BASKET_ID,SALES_VALUE,campaign_period
1,68,346,383,38,312,31625205750,0.34,BEFORE
1,68,346,383,38,312,31625206349,1.52,BEFORE
1,68,346,383,38,312,31625206349,1.29,BEFORE
1,68,346,383,38,312,31625205750,1.29,BEFORE
1,68,346,383,38,313,31672661958,0.89,BEFORE
1,68,346,383,38,313,31672661958,1.0,BEFORE
1,68,346,383,38,313,31672661958,0.99,BEFORE
1,68,346,383,38,313,31672661958,1.29,BEFORE
1,68,346,383,38,313,31672661958,2.0,BEFORE
1,68,346,383,38,313,31672661958,3.19,BEFORE


### 3. Aggregate Pre-Campaign and Campaign-Period Behavior

Aggregate transaction activity for campaign-assigned households across the equally sized BEFORE and DURING periods.

The comparison focuses on:
- total sales value,
- distinct baskets,
- average basket value.

**Target Grain:** One row per campaign and comparison period.

In [0]:
WITH campaign_transactions AS (

    SELECT
        ch.CAMPAIGN,
        ch.household_key,
        t.DAY,
        t.BASKET_ID,
        t.SALES_VALUE,

        CASE
            WHEN t.DAY BETWEEN
                d.START_DAY - (d.END_DAY - d.START_DAY + 1)
                AND d.START_DAY - 1
            THEN 'BEFORE'

            WHEN t.DAY BETWEEN d.START_DAY AND d.END_DAY
            THEN 'DURING'

        END AS campaign_period

    FROM workspace.consumer_analytics_silver.campaign_households ch

    INNER JOIN workspace.consumer_analytics_silver.campaign_desc d
        ON ch.CAMPAIGN = d.CAMPAIGN

    INNER JOIN workspace.consumer_analytics_silver.transactions t
        ON ch.household_key = t.household_key

    WHERE t.DAY BETWEEN
        d.START_DAY - (d.END_DAY - d.START_DAY + 1)
        AND d.END_DAY
)

SELECT
    CAMPAIGN,
    campaign_period,

    ROUND(SUM(SALES_VALUE), 2) AS total_spend,

    COUNT(DISTINCT BASKET_ID) AS total_baskets,

    ROUND(
        SUM(SALES_VALUE) / COUNT(DISTINCT BASKET_ID),
        2
    ) AS avg_basket_value

FROM campaign_transactions

GROUP BY
    CAMPAIGN,
    campaign_period

ORDER BY
    CAMPAIGN,
    campaign_period;

CAMPAIGN,campaign_period,total_spend,total_baskets,avg_basket_value
1,BEFORE,3223.57,110,29.31
1,DURING,3753.24,141,26.62
2,BEFORE,11568.4,408,28.35
2,DURING,11595.22,401,28.92
3,BEFORE,8757.21,257,34.07
3,DURING,8557.36,263,32.54
4,BEFORE,25407.06,823,30.87
4,DURING,26521.71,774,34.27
5,BEFORE,70996.56,1966,36.11
5,DURING,71934.0,1944,37.0


### 4. Calculate Campaign-Period Behavior Changes

Compare purchasing behavior during each campaign with the equally sized pre-campaign period.

The analysis measures percentage changes in:
- total spend,
- basket count,
- average basket value.

These metrics describe behavioral changes observed during campaign periods but do not establish causal campaign impact.

**Target Grain:** One row per campaign.

In [0]:
WITH campaign_transactions AS (

    SELECT
        ch.CAMPAIGN,
        t.BASKET_ID,
        t.SALES_VALUE,

        CASE
            WHEN t.DAY BETWEEN
                d.START_DAY - (d.END_DAY - d.START_DAY + 1)
                AND d.START_DAY - 1
            THEN 'BEFORE'

            WHEN t.DAY BETWEEN d.START_DAY AND d.END_DAY
            THEN 'DURING'
        END AS campaign_period

    FROM workspace.consumer_analytics_silver.campaign_households ch

    INNER JOIN workspace.consumer_analytics_silver.campaign_desc d
        ON ch.CAMPAIGN = d.CAMPAIGN

    INNER JOIN workspace.consumer_analytics_silver.transactions t
        ON ch.household_key = t.household_key

    WHERE t.DAY BETWEEN
        d.START_DAY - (d.END_DAY - d.START_DAY + 1)
        AND d.END_DAY
),

period_metrics AS (

    SELECT
        CAMPAIGN,
        campaign_period,
        SUM(SALES_VALUE) AS total_spend,
        COUNT(DISTINCT BASKET_ID) AS total_baskets,
        SUM(SALES_VALUE) / COUNT(DISTINCT BASKET_ID) AS avg_basket_value

    FROM campaign_transactions

    GROUP BY
        CAMPAIGN,
        campaign_period
),

campaign_comparison AS (

    SELECT
        CAMPAIGN,

        MAX(CASE WHEN campaign_period = 'BEFORE'
            THEN total_spend END) AS before_spend,

        MAX(CASE WHEN campaign_period = 'DURING'
            THEN total_spend END) AS during_spend,

        MAX(CASE WHEN campaign_period = 'BEFORE'
            THEN total_baskets END) AS before_baskets,

        MAX(CASE WHEN campaign_period = 'DURING'
            THEN total_baskets END) AS during_baskets,

        MAX(CASE WHEN campaign_period = 'BEFORE'
            THEN avg_basket_value END) AS before_avg_basket_value,

        MAX(CASE WHEN campaign_period = 'DURING'
            THEN avg_basket_value END) AS during_avg_basket_value

    FROM period_metrics

    GROUP BY CAMPAIGN
)

SELECT
    CAMPAIGN,

    ROUND(before_spend, 2) AS before_spend,
    ROUND(during_spend, 2) AS during_spend,

    ROUND(
        100.0 * (during_spend - before_spend)
        / NULLIF(before_spend, 0),
        2
    ) AS spend_change_pct,

    before_baskets,
    during_baskets,

    ROUND(
        100.0 * (during_baskets - before_baskets)
        / NULLIF(before_baskets, 0),
        2
    ) AS basket_change_pct,

    ROUND(before_avg_basket_value, 2) AS before_avg_basket_value,
    ROUND(during_avg_basket_value, 2) AS during_avg_basket_value,

    ROUND(
        100.0 * (during_avg_basket_value - before_avg_basket_value)
        / NULLIF(before_avg_basket_value, 0),
        2
    ) AS avg_basket_value_change_pct

FROM campaign_comparison

ORDER BY spend_change_pct DESC;

CAMPAIGN,before_spend,during_spend,spend_change_pct,before_baskets,during_baskets,basket_change_pct,before_avg_basket_value,during_avg_basket_value,avg_basket_value_change_pct
1,3223.57,3753.24,16.43,110,141,28.18,29.31,26.62,-9.17
21,27424.49,29945.72,9.19,585,592,1.20,46.88,50.58,7.9
20,198549.45,215726.02,8.65,5936,5719,-3.66,33.45,37.72,12.77
18,493929.73,533268.61,7.96,16747,16669,-0.47,29.49,31.99,8.47
19,44761.02,47669.13,6.5,1428,1327,-7.07,31.35,35.92,14.6
26,102603.84,107505.13,4.78,3467,3495,0.81,29.59,30.76,3.94
4,25407.06,26521.71,4.39,823,774,-5.95,30.87,34.27,11.0
28,10925.83,11330.83,3.71,268,228,-14.93,40.77,49.7,21.9
22,104987.06,107776.48,2.66,3084,3055,-0.94,34.04,35.28,3.63
7,72057.21,73966.46,2.65,2197,2388,8.69,32.8,30.97,-5.56


### 5. Build Campaign Behavior Signals Gold Table

Persist campaign-period behavioral comparisons as a reusable Gold analytical data product.

The table compares purchasing behavior of campaign-assigned households during each campaign with an equally sized period immediately before the campaign.

The metrics represent **observed behavioral changes** and should not be interpreted as causal campaign impact.

**Target Grain:** One row per campaign.

In [0]:
CREATE OR REPLACE TABLE workspace.consumer_analytics_gold.campaign_behavior_signals
USING DELTA
AS

WITH campaign_transactions AS (

    SELECT
        ch.CAMPAIGN,
        t.BASKET_ID,
        t.SALES_VALUE,

        CASE
            WHEN t.DAY BETWEEN
                d.START_DAY - (d.END_DAY - d.START_DAY + 1)
                AND d.START_DAY - 1
            THEN 'BEFORE'

            WHEN t.DAY BETWEEN d.START_DAY AND d.END_DAY
            THEN 'DURING'
        END AS campaign_period

    FROM workspace.consumer_analytics_silver.campaign_households ch

    INNER JOIN workspace.consumer_analytics_silver.campaign_desc d
        ON ch.CAMPAIGN = d.CAMPAIGN

    INNER JOIN workspace.consumer_analytics_silver.transactions t
        ON ch.household_key = t.household_key

    WHERE t.DAY BETWEEN
        d.START_DAY - (d.END_DAY - d.START_DAY + 1)
        AND d.END_DAY
),

period_metrics AS (

    SELECT
        CAMPAIGN,
        campaign_period,

        SUM(SALES_VALUE) AS total_spend,

        COUNT(DISTINCT BASKET_ID) AS total_baskets,

        SUM(SALES_VALUE)
            / COUNT(DISTINCT BASKET_ID) AS avg_basket_value

    FROM campaign_transactions

    GROUP BY
        CAMPAIGN,
        campaign_period
),

campaign_comparison AS (

    SELECT
        CAMPAIGN,

        MAX(CASE WHEN campaign_period = 'BEFORE'
            THEN total_spend END) AS before_spend,

        MAX(CASE WHEN campaign_period = 'DURING'
            THEN total_spend END) AS during_spend,

        MAX(CASE WHEN campaign_period = 'BEFORE'
            THEN total_baskets END) AS before_baskets,

        MAX(CASE WHEN campaign_period = 'DURING'
            THEN total_baskets END) AS during_baskets,

        MAX(CASE WHEN campaign_period = 'BEFORE'
            THEN avg_basket_value END) AS before_avg_basket_value,

        MAX(CASE WHEN campaign_period = 'DURING'
            THEN avg_basket_value END) AS during_avg_basket_value

    FROM period_metrics

    GROUP BY CAMPAIGN
)

SELECT
    CAMPAIGN,

    ROUND(before_spend, 2) AS before_spend,
    ROUND(during_spend, 2) AS during_spend,

    ROUND(
        100.0 * (during_spend - before_spend)
        / NULLIF(before_spend, 0),
        2
    ) AS spend_change_pct,

    before_baskets,
    during_baskets,

    ROUND(
        100.0 * (during_baskets - before_baskets)
        / NULLIF(before_baskets, 0),
        2
    ) AS basket_change_pct,

    ROUND(before_avg_basket_value, 2)
        AS before_avg_basket_value,

    ROUND(during_avg_basket_value, 2)
        AS during_avg_basket_value,

    ROUND(
        100.0 * (
            during_avg_basket_value - before_avg_basket_value
        )
        / NULLIF(before_avg_basket_value, 0),
        2
    ) AS avg_basket_value_change_pct,

    CURRENT_TIMESTAMP() AS _created_at

FROM campaign_comparison;

num_affected_rows,num_inserted_rows


### 6. Validate Campaign Behavior Signals

Validate that the final Gold table preserves the expected campaign-level grain and contains complete behavioral comparison metrics.

**Expected Grain:** One row per campaign.

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CAMPAIGN) AS distinct_campaigns

FROM workspace.consumer_analytics_gold.campaign_behavior_signals;

total_rows,distinct_campaigns
30,30


### 7. Validate Behavioral Metrics

Check the campaign behavior signals table for missing or invalid comparison metrics.

The validation ensures that BEFORE and DURING observations are available for every campaign and that core purchasing metrics contain valid values.

In [0]:
SELECT
    SUM(CASE WHEN before_spend IS NULL THEN 1 ELSE 0 END)
        AS null_before_spend,

    SUM(CASE WHEN during_spend IS NULL THEN 1 ELSE 0 END)
        AS null_during_spend,

    SUM(CASE WHEN before_baskets IS NULL THEN 1 ELSE 0 END)
        AS null_before_baskets,

    SUM(CASE WHEN during_baskets IS NULL THEN 1 ELSE 0 END)
        AS null_during_baskets,

    SUM(CASE WHEN spend_change_pct IS NULL THEN 1 ELSE 0 END)
        AS null_spend_change,

    SUM(CASE WHEN basket_change_pct IS NULL THEN 1 ELSE 0 END)
        AS null_basket_change,

    SUM(CASE WHEN avg_basket_value_change_pct IS NULL THEN 1 ELSE 0 END)
        AS null_basket_value_change,

    SUM(CASE WHEN before_spend < 0 OR during_spend < 0 THEN 1 ELSE 0 END)
        AS negative_spend,

    SUM(CASE WHEN before_baskets <= 0 OR during_baskets <= 0 THEN 1 ELSE 0 END)
        AS invalid_basket_count

FROM workspace.consumer_analytics_gold.campaign_behavior_signals;

null_before_spend,null_during_spend,null_before_baskets,null_during_baskets,null_spend_change,null_basket_change,null_basket_value_change,negative_spend,invalid_basket_count
0,0,0,0,0,0,0,0,0


### 8. Combine Campaign Performance and Behavior Signals

Combine campaign response metrics with observed purchasing behavior changes to support campaign-level exploratory analysis.

The combined view allows redemption activity and BEFORE-vs-DURING purchasing patterns to be evaluated together.

Observed relationships are descriptive and should not be interpreted as causal campaign effects.

In [0]:
SELECT
    p.CAMPAIGN,
    p.DESCRIPTION,

    p.households_assigned,
    p.households_redeemed,
    p.redemption_rate_pct,

    s.spend_change_pct,
    s.basket_change_pct,
    s.avg_basket_value_change_pct

FROM workspace.consumer_analytics_gold.campaign_performance p

INNER JOIN workspace.consumer_analytics_gold.campaign_behavior_signals s
    ON p.CAMPAIGN = s.CAMPAIGN

ORDER BY s.spend_change_pct DESC;

CAMPAIGN,DESCRIPTION,households_assigned,households_redeemed,redemption_rate_pct,spend_change_pct,basket_change_pct,avg_basket_value_change_pct
1,TypeB,13,1,7.69,16.43,28.18,-9.17
21,TypeB,65,4,6.15,9.19,1.20,7.9
20,TypeC,244,20,8.20,8.65,-3.66,12.77
18,TypeA,1133,214,18.89,7.96,-0.47,8.47
19,TypeB,130,15,11.54,6.5,-7.07,14.6
26,TypeA,332,31,9.34,4.78,0.81,3.94
4,TypeB,81,6,7.41,4.39,-5.95,11.0
28,TypeB,17,1,5.88,3.71,-14.93,21.9
22,TypeB,276,17,6.16,2.66,-0.94,3.63
7,TypeB,198,5,2.53,2.65,8.69,-5.56


### 9. Create Campaign Analytical Summary

Create a consolidated campaign-level analytical view combining campaign response metrics with observed purchasing behavior changes.

This output supports exploratory campaign analysis by presenting redemption, spending, shopping frequency, and average basket value metrics together.

The behavioral change metrics are descriptive comparisons between equally sized BEFORE and DURING periods and should not be interpreted as causal campaign effects.

**Target Grain:** One row per campaign.

In [0]:
CREATE OR REPLACE VIEW workspace.consumer_analytics_gold.campaign_analytical_summary
AS

SELECT
    p.CAMPAIGN,
    p.DESCRIPTION,
    p.START_DAY,
    p.END_DAY,
    p.campaign_duration_days,

    p.households_assigned,
    p.households_redeemed,
    p.redemption_events,
    p.redemption_rate_pct,

    s.before_spend,
    s.during_spend,
    s.spend_change_pct,

    s.before_baskets,
    s.during_baskets,
    s.basket_change_pct,

    s.before_avg_basket_value,
    s.during_avg_basket_value,
    s.avg_basket_value_change_pct

FROM workspace.consumer_analytics_gold.campaign_performance p

INNER JOIN workspace.consumer_analytics_gold.campaign_behavior_signals s
    ON p.CAMPAIGN = s.CAMPAIGN;

### 10. Validate Campaign Analytical Summary

Validate that the final analytical view preserves the expected campaign-level grain.

**Expected:** 30 campaigns with one row per campaign.

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CAMPAIGN) AS distinct_campaigns
FROM workspace.consumer_analytics_gold.campaign_analytical_summary;

total_rows,distinct_campaigns
30,30


### 11. Validate Campaign Observation Windows

Check whether each campaign's full active period is covered by the available transaction history.

Campaigns extending beyond the maximum transaction day have incomplete DURING observation windows. Their BEFORE-vs-DURING changes should therefore not be directly interpreted alongside campaigns with complete observation windows.

In [0]:
SELECT
    d.CAMPAIGN,
    d.START_DAY,
    d.END_DAY,
    d.END_DAY - d.START_DAY + 1 AS campaign_duration_days,
    m.max_transaction_day,

    CASE
        WHEN d.END_DAY <= m.max_transaction_day
        THEN 'COMPLETE'
        ELSE 'INCOMPLETE'
    END AS observation_status

FROM workspace.consumer_analytics_silver.campaign_desc d

CROSS JOIN (
    SELECT MAX(DAY) AS max_transaction_day
    FROM workspace.consumer_analytics_silver.transactions
) m

ORDER BY d.CAMPAIGN;

CAMPAIGN,START_DAY,END_DAY,campaign_duration_days,max_transaction_day,observation_status
1,346,383,38,711,COMPLETE
2,351,383,33,711,COMPLETE
3,356,412,57,711,COMPLETE
4,372,404,33,711,COMPLETE
5,377,411,35,711,COMPLETE
6,393,425,33,711,COMPLETE
7,398,432,35,711,COMPLETE
8,412,460,49,711,COMPLETE
9,435,467,33,711,COMPLETE
10,463,495,33,711,COMPLETE


### 12. Add Observation Window Status

Add an observation status to the final analytical view to distinguish campaigns with complete transaction coverage from campaigns whose active period extends beyond the available transaction history.

Incomplete campaigns are retained for transparency but should be excluded from direct BEFORE-vs-DURING performance comparisons.

In [0]:
CREATE OR REPLACE VIEW workspace.consumer_analytics_gold.campaign_analytical_summary
AS

WITH transaction_coverage AS (
    SELECT
        MAX(DAY) AS max_transaction_day
    FROM workspace.consumer_analytics_silver.transactions
)

SELECT
    p.CAMPAIGN,
    p.DESCRIPTION,
    p.START_DAY,
    p.END_DAY,
    p.campaign_duration_days,

    CASE
        WHEN p.END_DAY <= tc.max_transaction_day
        THEN 'COMPLETE'
        ELSE 'INCOMPLETE'
    END AS observation_status,

    p.households_assigned,
    p.households_redeemed,
    p.redemption_events,
    p.redemption_rate_pct,

    s.before_spend,
    s.during_spend,
    s.spend_change_pct,

    s.before_baskets,
    s.during_baskets,
    s.basket_change_pct,

    s.before_avg_basket_value,
    s.during_avg_basket_value,
    s.avg_basket_value_change_pct

FROM workspace.consumer_analytics_gold.campaign_performance p

INNER JOIN workspace.consumer_analytics_gold.campaign_behavior_signals s
    ON p.CAMPAIGN = s.CAMPAIGN

CROSS JOIN transaction_coverage tc;

In [0]:
SELECT
    observation_status,
    COUNT(*) AS campaigns
FROM workspace.consumer_analytics_gold.campaign_analytical_summary
GROUP BY observation_status;

observation_status,campaigns
INCOMPLETE,1
COMPLETE,29
